# Walk-forward trading test: learn from the past, measure profit on the future
The model only ever sees older data. Every month it is tested on the **next** month it has never seen, with trading fees and slippage included.
Accuracy here = **net profit in %** on unseen data. A final 25% of the history is kept untouched and scored exactly once, after the settings are chosen.
**PASS** = positive net profit, at least 30 trades spread over at least 4 of 6 slices of the holdout, no single slice giving more than half the profit, and better than random entries (p < 0.05). This is research, not financial advice. A pass can still lose money live.
Run: Runtime > Run all (CPU is fine, about 15-30 min).

In [ ]:
SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT', 'BNBUSDT', 'DOGEUSDT', 'ADAUSDT', 'LINKUSDT', 'AVAXUSDT', 'LTCUSDT']
GRAN = '4h'            # candle size: '1h' or '4h'
DAYS = 1500              # history to download
FEE = 0.001             # 0.1% per side (Bitget spot taker)
SLIP = 0.0005           # 0.05% slippage per round trip
TARGET_PCT = 20.0       # stop searching settings when walk-forward profit on the DEV part reaches this
MAX_CONFIGS = 15
HOLDS = (3, 6, 12, 24, 48)   # trade hold times in candles (4h candles: 12h up to 8 days)
MIN_TRADES = 30         # holdout needs at least this many trades
MIN_ACTIVE_SEGMENTS = 4 # ...spread over at least 4 of 6 equal slices of the holdout

In [ ]:
import numpy as np, pandas as pd, json, time, itertools, random
from sklearn.ensemble import HistGradientBoostingClassifier
BAR_H = 1   # hours per candle (set by learn())

# ---------------------------------------------------------------- data
def _get(url, params, tries=4):
    import requests
    for i in range(tries):
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.status_code == 200: return r.json()
        except Exception as e:
            err = e
        time.sleep(1 + i)
    return None

def fetch_bitget(symbol, gran='1h', days=900):
    ms_bar = {'1h': 3600_000, '4h': 4 * 3600_000, '15min': 900_000}[gran]
    end = int(time.time() * 1000); stop = end - days * 86400_000; rows = []
    while end > stop:
        j = _get('https://api.bitget.com/api/v2/spot/market/history-candles',
                 dict(symbol=symbol, granularity=gran, endTime=end, limit=200))
        if not j or j.get('code') != '00000' or not j.get('data'): break
        b = [[int(x[0]), *map(float, x[1:6])] for x in j['data']]
        rows += b; new_end = min(x[0] for x in b) - 1
        if new_end >= end: break
        end = new_end; time.sleep(0.12)
    return _frame(rows)

def fetch_binance(symbol, gran='1h', days=900):
    end = int(time.time() * 1000); stop = end - days * 86400_000; rows = []
    while end > stop:
        j = _get('https://data-api.binance.vision/api/v3/klines',
                 dict(symbol=symbol, interval=gran, endTime=end, limit=1000))
        if not j: break
        b = [[int(x[0]), float(x[1]), float(x[2]), float(x[3]), float(x[4]), float(x[5])] for x in j]
        rows += b; new_end = min(x[0] for x in b) - 1
        if new_end >= end: break
        end = new_end; time.sleep(0.1)
    return _frame(rows)

def _frame(rows):
    if not rows: return None
    d = pd.DataFrame(rows, columns=['ts', 'open', 'high', 'low', 'close', 'vol']).drop_duplicates('ts').sort_values('ts')
    d.index = pd.to_datetime(d.ts, unit='ms'); return d.drop(columns='ts')

def synthetic(n=20000, seed=0, edge=0.0, freq='h'):
    """random-walk candles with volatility clustering; edge>0 injects a real, learnable momentum effect."""
    rng = np.random.default_rng(seed); vol = np.empty(n); v = 0.004
    for i in range(n): v = 0.98 * v + 0.02 * 0.004 + 0.0006 * abs(rng.normal()); vol[i] = v
    z = rng.normal(size=n); r = vol * z
    if edge:
        for i in range(24, n):
            m6 = r[i - 6:i].sum()
            if abs(m6) > 1.5 * vol[i]: r[i] += edge * vol[i] * np.sign(m6)
    c = 100 * np.exp(np.cumsum(r)); o = np.r_[c[0], c[:-1]]
    hi = np.maximum(o, c) * (1 + vol * abs(rng.normal(size=n)) * .3); lo = np.minimum(o, c) * (1 - vol * abs(rng.normal(size=n)) * .3)
    idx = pd.date_range('2022-01-01', periods=n, freq=freq)
    return pd.DataFrame(dict(open=o, high=hi, low=lo, close=c, vol=rng.lognormal(5, .5, n)), index=idx)

# ---------------------------------------------------------------- features
FEATS = ['r1', 'r3', 'r6', 'r12', 'r24', 'r72', 'vol24', 'vol72', 'rsi14', 'ema24', 'ema96', 'volz', 'rng', 'hs', 'hc', 'dow']
def features(d):
    c = d.close; lr = np.log(c).diff(); f = pd.DataFrame(index=d.index)
    for k in (1, 3, 6, 12, 24, 72): f['r%d' % k] = np.log(c).diff(k)
    f['vol24'] = lr.rolling(24).std(); f['vol72'] = lr.rolling(72).std()
    up = lr.clip(lower=0).rolling(14).mean(); dn = (-lr.clip(upper=0)).rolling(14).mean(); f['rsi14'] = up / (up + dn + 1e-12)
    f['ema24'] = c / c.ewm(span=24).mean() - 1; f['ema96'] = c / c.ewm(span=96).mean() - 1
    f['volz'] = d.vol / d.vol.rolling(72).mean() - 1; f['rng'] = (d.high - d.low) / c
    h = d.index.hour; f['hs'] = np.sin(2 * np.pi * h / 24); f['hc'] = np.cos(2 * np.pi * h / 24); f['dow'] = d.index.dayofweek
    return f[FEATS]

def prep(data, H):
    """per symbol: features at bar close t, trade = buy open[t+1], sell open[t+1+H]."""
    out = {}
    for s, d in data.items():
        f = features(d); o = d.open.values
        n = len(d); gross = np.full(n, np.nan)
        gross[:n - H - 1] = o[1 + H:n] / o[1:n - H] - 1
        f = f.copy(); f['gross'] = gross; out[s] = f.dropna(subset=FEATS)
    return out

# ---------------------------------------------------------------- model + simulation
def fit(X, y, depth, seed=0):
    m = HistGradientBoostingClassifier(max_depth=depth, learning_rate=0.05, max_iter=100, l2_regularization=5.0,
                                       min_samples_leaf=200, random_state=seed)
    return m.fit(X, y)

def trades(df, p, thr, H):
    """non-overlapping long trades; returns (entry_index_positions, gross returns)."""
    sel = np.flatnonzero((p > thr) & df.gross.notna().values); pos = []; nxt = -1
    for i in sel:
        if i >= nxt: pos.append(i); nxt = i + 1 + H
    return np.array(pos, int)

def sim(per_sym, probs, thr, H, cost):
    """returns list of (exit_time, net_return, symbol)."""
    out = []
    for s, df in per_sym.items():
        pos = trades(df, probs[s], thr, H)
        for i in pos: out.append((df.index[i] + pd.Timedelta(hours=BAR_H * (1 + H)), df.gross.iloc[i] - cost, s))
    return out

def pick_thr(per_sym_val, probs_val, H, cost, grid=np.arange(0.50, 0.76, 0.02), min_trades=15):
    best = (0.0, 1.01)  # default: do not trade
    for t in grid:
        tr = sim(per_sym_val, probs_val, t, H, cost)
        if len(tr) >= min_trades:
            r = np.mean([x[1] for x in tr])
            if r > best[0]: best = (r, float(t))
    return best[1]

def walk_forward(data, cfg, start, end, cost, min_train_days=270, test_days=45, log=False, score_from=None):
    """expanding-window walk-forward on data[start:end]. every test month is predicted by a model that only saw the past."""
    H, depth = cfg['H'], cfg['depth']; P = prep(data, H); allr = []
    idx = sorted(set().union(*[set(df.index) for df in P.values()])); idx = [t for t in idx if start <= t < end]
    t0 = idx[0] + pd.Timedelta(days=min_train_days)
    while t0 < idx[-1]:
        t1 = t0 + pd.Timedelta(days=test_days); gap = pd.Timedelta(hours=BAR_H * (H + 2))
        if score_from is not None and t1 <= score_from: t0 = t1; continue
        tr, te = {}, {}
        for s, df in P.items():
            tr[s] = df[(df.index >= start) & (df.index < t0 - gap)]; te[s] = df[(df.index >= t0) & (df.index < min(t1, end))]
        if min(len(v) for v in te.values()) < 24: break
        # inner validation = last 20% of the training window (threshold chosen here, never on the test month)
        cut = {s: v.index[int(len(v) * 0.8)] for s, v in tr.items()}
        fitX = pd.concat([v[v.index < cut[s]] for s, v in tr.items()]); gap2 = pd.Timedelta(hours=BAR_H * (H + 2))
        y = (fitX.gross > cost).astype(int)
        m = fit(fitX[FEATS], y, depth)
        val = {s: v[v.index >= cut[s] + gap2] for s, v in tr.items()}
        pv = {s: m.predict_proba(v[FEATS])[:, 1] for s, v in val.items()}
        thr = pick_thr(val, pv, H, cost)
        # final model for the test month: refit on the whole (purged) training window
        allX = pd.concat(list(tr.values())); m2 = fit(allX[FEATS], (allX.gross > cost).astype(int), depth)
        pt = {s: m2.predict_proba(v[FEATS])[:, 1] for s, v in te.items()}
        res = sim(te, pt, thr, H, cost); allr += res
        if log: print(' ', t0.date(), 'thr', round(thr, 2), 'trades', len(res), 'net %.2f%%' % (100 * np.sum([x[1] for x in res]) / max(1, len(data))))
        t0 = t1
    return allr, P

def summarize(trs, data, start, end, label=''):
    """portfolio: equal capital per symbol, each compounding its own trades. returns dict (percent numbers)."""
    syms = list(data); eq = {}
    for s in syms:
        r = sorted([(t, x) for t, x, ss in trs if ss == s])
        eq[s] = pd.Series([1 + x for _, x in r], index=[t for t, _ in r]).cumprod() if r else pd.Series(dtype=float)
    grid = pd.date_range(start, end, freq='D'); cols = []
    for s in syms:
        e = eq[s]; cols.append(e.reindex(grid, method='ffill').fillna(1.0) if len(e) else pd.Series(1.0, index=grid))
    port = pd.concat(cols, axis=1).mean(axis=1); dd = (port / port.cummax() - 1).min()
    rets = np.array([x for _, x, _ in trs]) if trs else np.array([])
    bh = np.mean([data[s].close[(data[s].index >= start) & (data[s].index < end)].pipe(lambda c: c.iloc[-1] / c.iloc[0] - 1) for s in syms])
    return dict(label=label, net_profit_pct=100 * (port.iloc[-1] - 1), buy_hold_pct=100 * bh, trades=len(rets),
                win_rate_pct=100 * float((rets > 0).mean()) if len(rets) else 0.0,
                avg_trade_pct=100 * float(rets.mean()) if len(rets) else 0.0, max_drawdown_pct=100 * dd, equity=port)

def random_baseline(P, H, cost, n_trades, mean_model, n=1000, seed=1):
    """how often do RANDOM entries (same count, same hold time, same costs) match the model's average trade? returns p-value."""
    rng = np.random.default_rng(seed); g = np.concatenate([df.gross.dropna().values for df in P.values()]) - cost
    if n_trades == 0 or len(g) == 0: return 1.0
    m = np.array([rng.choice(g, n_trades).mean() for _ in range(n)]); return float((m >= mean_model).mean())

# ---------------------------------------------------------------- the learning loop
def learn(data, FEE=0.001, SLIP=0.0005, TARGET_PCT=20.0, MAX_CONFIGS=15, HOLDOUT=0.25, GRAN='4h', HOLDS=(3, 6, 12, 24, 48), log=True, seed=0,
          MIN_TRADES=30, MIN_ACTIVE_SEGMENTS=4, MAX_TOP_SEGMENT_SHARE=50.0):
    global BAR_H
    BAR_H = {'1h': 1, '4h': 4, '15min': 0.25}[GRAN]; cost = 2 * FEE + SLIP
    ts = sorted(set().union(*[set(d.index) for d in data.values()])); t_start, t_end = ts[0], ts[-1] + pd.Timedelta(hours=BAR_H)
    t_hold = t_start + (t_end - t_start) * (1 - HOLDOUT)
    grid = [dict(H=H, depth=dp) for H in HOLDS for dp in (2, 3, 4)]; random.Random(seed).shuffle(grid)
    hist = []; best = None
    for k, cfg in enumerate(grid[:MAX_CONFIGS]):
        trs, _ = walk_forward(data, cfg, t_start, t_hold, cost)
        sm = summarize(trs, data, t_start + pd.Timedelta(days=270), t_hold, str(cfg)); hist.append((cfg, sm))
        if log: print('config %d/%d %s (hold %.0fh) -> dev walk-forward net profit %.1f%% (%d trades, win %.0f%%, maxDD %.1f%%)' %
                      (k + 1, MAX_CONFIGS, cfg, cfg['H'] * BAR_H, sm['net_profit_pct'], sm['trades'], sm['win_rate_pct'], sm['max_drawdown_pct']), flush=True)
        if sm['trades'] >= MIN_TRADES and (best is None or sm['net_profit_pct'] > best[1]['net_profit_pct']): best = (cfg, sm)
        if best is not None and best[1]['net_profit_pct'] >= TARGET_PCT:
            if log: print('dev target reached'); break
    if best is None: best = max(hist, key=lambda x: x[1]['net_profit_pct'])
    cfg = best[0]
    # ---- the honest test: untouched holdout, evaluated exactly once
    trs, P = walk_forward(data, cfg, t_start, t_end, cost, score_from=t_hold)
    trs = [x for x in trs if x[0] >= t_hold]
    sm = summarize(trs, data, t_hold, t_end, 'HOLDOUT')
    pv = random_baseline({s: df[df.index >= t_hold] for s, df in P.items()}, cfg['H'], cost, sm['trades'], sm['avg_trade_pct'] / 100)
    # spread rule: trades must be spread over the holdout, not one lucky burst
    nseg = 6; edges = [t_hold + (t_end - t_hold) * i / nseg for i in range(nseg + 1)]
    seg_profit = []; seg_trades = []
    for i in range(nseg):
        r = [x for t, x, _ in trs if edges[i] <= t < edges[i + 1]]; seg_trades.append(len(r)); seg_profit.append(100 * sum(r))
    pos = [p for p in seg_profit if p > 0]; tot_pos = sum(pos)
    top_share = 100 * max(pos) / tot_pos if tot_pos > 0 else 100.0
    sm.update(p_value_vs_random=pv, seg_trades=seg_trades, seg_profit=seg_profit, active_segments=sum(1 for n in seg_trades if n > 0), top_segment_share_pct=top_share)
    checks = dict(profit=sm['net_profit_pct'] > 0, trades=sm['trades'] >= MIN_TRADES, spread=sm['active_segments'] >= MIN_ACTIVE_SEGMENTS,
                  not_one_burst=top_share <= MAX_TOP_SEGMENT_SHARE, beats_random=pv < 0.05)
    sm['checks'] = checks; sm['passes'] = all(checks.values())
    return cfg, best[1], sm, hist, dict(t_start=t_start, t_hold=t_hold, t_end=t_end, cost=cost)

In [ ]:
data = {}
for s in SYMBOLS:
    d = fetch_bitget(s, GRAN, DAYS)
    if d is None or len(d) < 2000: d = fetch_binance(s, GRAN, DAYS); src = 'binance'
    else: src = 'bitget'
    if d is None or len(d) < 2000: print(s, 'download failed'); continue
    data[s] = d; print(s, src, len(d), 'candles', d.index[0].date(), '->', d.index[-1].date())
assert len(data) >= 1, 'no data downloaded, tell Claude the messages above'

In [ ]:
cfg, dev, sm, hist, info = learn(data, FEE, SLIP, TARGET_PCT, MAX_CONFIGS, GRAN=GRAN, HOLDS=HOLDS, MIN_TRADES=MIN_TRADES, MIN_ACTIVE_SEGMENTS=MIN_ACTIVE_SEGMENTS)
print()
print('chosen settings:', cfg, '(hold %.0f hours per trade)' % (cfg['H'] * BAR_H))
print('DEV (walk-forward, used to choose settings): net profit %.1f%%' % dev['net_profit_pct'])
print('=' * 60)
print('HOLDOUT (never seen, scored once): %s .. %s' % (info['t_hold'].date(), info['t_end'].date()))
print('  net profit after fees : %.1f %%' % sm['net_profit_pct'])
print('  buy & hold same period: %.1f %%' % sm['buy_hold_pct'])
print('  trades %d | win rate %.0f%% | avg trade %.3f%% | max drawdown %.1f%%' % (sm['trades'], sm['win_rate_pct'], sm['avg_trade_pct'], sm['max_drawdown_pct']))
print('  p-value vs random entries: %.3f (below 0.05 = unlikely to be luck)' % sm['p_value_vs_random'])
print('  trades per holdout slice (6 slices):', sm['seg_trades'], '| profit % per slice:', [round(x, 1) for x in sm['seg_profit']])
print('  share of profit from best slice: %.0f%% (must be <= 50%%)' % sm['top_segment_share_pct'])
print('  checks:', {k: bool(v) for k, v in sm['checks'].items()})
print('  RESULT:', 'PASS' if sm['passes'] else 'FAIL - no reliable edge found, do not trade this')
import matplotlib.pyplot as plt
e = sm['equity']; plt.figure(figsize=(9, 3.5)); plt.plot(e.index, (e - 1) * 100); plt.axhline(0, color='gray'); plt.title('Holdout equity, % profit'); plt.show()
json.dump({k: v for k, v in sm.items() if k != 'equity'} | {'cfg': cfg}, open('report.json', 'w'), indent=1, default=float)